# Project: Random Forests
## Name: Jesed Renteria

In [ ]:
#Example of supress warnings for Numpy version out of range (optional)
import warnings
warnings.filterwarnings("ignore", category=Warning)
warnings.simplefilter(action='ignore', category=FutureWarning)

#Some recommended libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import learning_curve
import seaborn as sns

# The Dataset

In [ ]:
# Load the Wine Quality dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, delimiter=";")
df.head()
df.info()

# Data Preprocessing

In [ ]:
#Insert Code Here
print(df.describe())
corr_matrix = df.corr()
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title('Correlation Matrix')
plt.show()
#Volatile acidity and alcohol seem to have a strong correlation with quality

In [ ]:
#Standardization & Train-Test-Split
scaler = StandardScaler()
X = df.drop(columns = ["quality"])
y = df["quality"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

# Building the Random Forest Model

In [ ]:
#Insert Code Here
# Choosing Hyperparameters:
# n_estimators is the number of trees in the forest
# max_depth controls the maximum depth the tree is allowed to grow to
# min_samples_split controls the minimum number of samples required to split a node
# max_features are the number of features to consider when looking for the best split

clf_rf = RandomForestClassifier(random_state=42, max_depth = 5, min_samples_split = 7, max_features ='sqrt', n_estimators = 200)
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)


# Evaluating the Model

In [ ]:
#Insert Code Here
accuracy = accuracy_score(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf, average = "weighted")
recall = recall_score(y_test, y_pred_rf, average = "weighted")
f1 = f1_score(y_test, y_pred_rf, average = "weighted")
conf_matrix = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix:")
print(conf_matrix, "\n")
print(f"Accuracy: {accuracy}", '\n')
print(f"Precision: {precision}", '\n')
print(f"Recall: {recall}", '\n')
print(f"F1 Score: {f1}", '\n')

plt.figure(figsize=(8,6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title(f"Confusion Matrix - Random Forest Classifier (Accuracy: {accuracy:.2f})")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## For the Model Selection Project, you will STOP HERE! 
During Units 4, 5, and 6, we will explore and learn additional techniques, and then revisit these projects to apply the below:
- Model evaluation and parameter tuning
- Explanatory visualizations and package your results with data storytelling

# Tuning Model Parameters (Completed in Unit 4)

In [ ]:
#Insert Code Here
#Get feature importances
importances = clf_rf.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = df.columns[:-1]

#Number of trees in the forest
#n_estimators = 100     

#Maximum depth of the tree
#max_depth = None       

#Minimum number of samples required to split an internal node
#min_samples_split = 2  

#Number of features to consider when looking for the best split
#max_features = 'sqrt' 
param_grid1= {
    "n_estimators" : [50, 100, 200],
    "max_depth" : [10, 15, 20, None],
    "min_samples_split" : [2, 5, 7, 10],
    "max_features" : ["sqrt", "log2"]
}

grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state = 42), param_grid=param_grid1, cv=5, scoring='accuracy', verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

# Evaluating the Tuned Model (Completed in Unit 4)

In [ ]:
#Insert Code Here
print("Best hyperparameters: ", grid_search.best_params_)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average = "weighted")
recall = recall_score(y_test, y_pred, average = "weighted")
f1 = f1_score(y_test, y_pred, average = "weighted")

conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix, "\n")
print(f"Accuracy: {accuracy}", '\n')
print(f"Precision: {precision}", '\n')
print(f"Recall: {recall}", '\n')
print(f"F1 Score: {f1}", '\n')

plt.figure(figsize=(8,6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title(f"Confusion Matrix - Random Forest Classifier (Accuracy: {accuracy:.2f})")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Visualizing Results (Completed in Units 4 and 6)

In [ ]:
#Insert Code Here
importances = clf_rf.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = df.columns[:-1]
plt.figure(figsize=(10, 6))
plt.title("Feature Importance in Wine Quality Prediction")
plt.barh(range(X.shape[1]), importances[indices], align="center")
plt.yticks(range(X.shape[1]), feature_names[indices])
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

In [ ]:
#Learning Curve Plot
plt.figure(figsize=(10, 6))
train_sizes, train_scores, test_scores = learning_curve(
    estimator = best_rf,
    X = X_train,
    y = y_train,
    train_sizes = np.linspace(0.1, 1.0, 5),
    cv = 5,
    scoring = "accuracy",
    n_jobs = -1
)
# Mean and Standard Deviations for Training and Testing subsets
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

# Plot using matplotlib
plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Accuracy')
plt.plot(train_sizes, test_mean, 'o-', color='green', label='Validation Accuracy')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='green')
plt.title('Learning Curve for Random Forest (Best Parameters)')
plt.xlabel('Number of Training Samples')
plt.ylabel('Accuracy')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Confusion Matrix Visualization
plt.figure(figsize=(8,6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title(f"Confusion Matrix - Random Forest Classifier (Accuracy: {accuracy:.2f})")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


    Audience:
    This presentation will aimed towards data scientists, winemakers, wine tasters, wineries or wine companies, and wine consumers. Winemakers, tasters and wineries or wine companies have knowledge in creating fine quality wine, however, it is difficult to find the best quality for certain people who enjoy wine and if they are able to improve their quality of their wine, lots of profits could be made and the tasters could enjoy their high quality wine once they learned what features impact the quality of wine. As for data scientists, they would intrigued to look at the model that was made and even improve the model to get a much accurate and efficient model.

    Introduction: 
    Finding what improves the quality of wine is a challenge as various ingredients go into wine to make up that quality. The Wine Quality dataset contains samples with features representing various chemical properties such as acidity, residual sugar, and alcohol content. The target variable is the quality rating on a scale from 0 to 10.


    Data Preprocessing:
    During the data preprocessing stage, the dataset was loaded into a dataset using the Pandas library in python. Then, the dataset was thoroughly checked to see if there were any null values using the ".info()" function call that displayed the information of the dataset such as features of the dataset, non-null count and the datatype. Next, visuals were made specifically the correlation matrix to provide an understanding of the target variable, quality, that rates the quality of wine on a scale from 0 to 10, using the matplotlib library and the ".corr()" function call that returns a dataframe representing the correlation matrix, and noticed that two features had pretty high correlation with the target variable, which are "volatile acidty" with a negative correlation and "alcohol" with a positive correlation. Then, the target variable, quality, was separated into a variable, and the rest of features into another variable. Then, the rest of the features was standardized, using the "StandardScaler()" function call from the Scikit-Learn library, to have a mean of 0 and a standard deviation of 1 for consistent training and to ensure that the model is not biased towards features with larger scales.

    Model Architecture:
    The model that was selected is the Random Forest from the Scikit-Learn library which is a supervised machine learning algorithm that combines multiple decision trees to make predictions, which is great for classification and regression tasks. The parameters within the Random Forest model are n_estimators which is the number of trees in the forest, max_depth which controls the maximum depth the tree is allowed to grow to, min_samples_split which controls the minimum number of samples required to split a node, and max_features which is the number of features to consider when looking for the best split. Setting up the parameters, max_depth was set to 5, min_samples_split was set to 7, max_features was set to "sqrt", and the n_estimators was set to 200.
    
    Training and Evaluation:
    The training involved splitting the dataset into training and testing sets to evaluate the model's performance on unseen data. 80% of the dataset goes to the training subset and 20% of it goes to the testing subsets, using the "train_test_split()" function call from the Scikit-Learn library. After building the model to predict quality of wine and getting the needed metrics to evaluate the model, the model had an accuracy of 56.88%, a precision of 52.61%, a recall of 56.88% and a F1 score of 53.05%.
    
    Parameter Tuning:
    To find the optimal hyperparameters and their outcomes, a grid search, from the Scikit-Learn library, was used to experiment different combinations of the the Random Forest model to find the best estimator. As a result, the best estimator had the max_depth parameter of 100, max_features of "sqrt", min_samples_split of 2, and n_estimators of 200. When comparing the metrics for the best estimator, it was noticed the metrics improved with an accuracy of 63.13%, a precision of 59.78%, a recall of 63.13%, and an F1 score of 61.1%.

    Conclusion:
    Since working with the Wine Quality dataset, it helped myself understand how to use the Random Forest Classifier and how to tune it to improve the model to predict the quality of wine. Future improvements of the model would be to research other techniques to improve the Random Forest model as well as parameters that comes with the Random Forest Model from the Scikit-Learn library. Anoter improvement would be how I preprocessed the data since there might have been some steps I have did first instead of in order.